In [ ]:
from torch.distributions import Gamma
import torch
import numpy as np
import netCDF4
import glob
import matplotlib.pyplot as plt

campaign = "COMMON_2011"
fileall = glob.glob(f"../disdrodb-data/DISDRODB/Processed/EPFL/{campaign}/L0B/??/L0B.{campaign}.??.*.V0.nc")
infile = fileall[2]
nc = netCDF4.Dataset(infile, "r")

for i in range(1):
    print(f"{i=}")
    # Synthetic histogram values (you should replace this with your actual histogram data)
    hist_values = np.sum(nc["raw_drop_number"][:], axis=2)[i][:20].astype(np.float32)
    hist_values_normalized = hist_values / (hist_values.sum() * bin_widths)
    #hist_values = np.random.poisson(10, size=len(bin_edges) - 1)

    # Compute bin centers and bin widths
    bin_centers = torch.tensor(np.array(nc["diameter_bin_center"][:][:20]))
    bin_widths = torch.tensor(np.array(nc["diameter_bin_width"][:][:20]))
    
    x_values = np.linspace(0, 5, 501)
    shape = 8
    mu = shape - 1
    scale = (1.935 + 0.735 * mu + 0.0365 * mu**2)
    fitted_pdf = Gamma(shape, scale).log_prob(x_values).exp()
    #fitted_pdf2 = gamma.pdf(x_values, a=fitted_shape2, scale=fitted_scale2)

    # Plot the histogram and the fitted Gamma PDF
    plt.figure(figsize=(8, 5))
    plt.bar(bin_centers, hist_values_normalized, width=bin_widths, alpha=0.5, label="Histogram")
    plt.plot(x_values, fitted_pdf, label=f"Fitted Gamma PDF\n(shape={shape:.2f}, scale={scale:.2f})", color="red")
    #plt.plot(x_values, fitted_pdf2, label=f"Fitted Gamma PDF\n(shape={fitted_shape2:.2f}, scale={fitted_scale2:.2f})", color="blue")
    #plt.annotate(f"{D1=:.2f} mm", [0.7, 0.65], xycoords='axes fraction', color="red")
    #plt.annotate(f"{D2=:.2f} mm", [0.7, 0.6], xycoords='axes fraction', color="blue")
    plt.xlim(0, 5)
    plt.xlabel("Value")
    plt.ylabel("Density")
    plt.legend()
    plt.title("EPFL, COMMON 2011")
    #plt.show()
    plt.savefig(f"step{i}")

In [ ]:
## from torch.distributions import Gamma
import torch
import numpy as np
import netCDF4
import glob

campaign = "COMMON_2011"
fileall = glob.glob(f"../disdrodb-data/DISDRODB/Processed/EPFL/{campaign}/L0B/??/L0B.{campaign}.??.*.V0.nc")
infile = fileall[2]
nc = netCDF4.Dataset(infile, "r")

def compute_negative_log_likelihood_vectorized(mu, bin_centers, bin_widths, hist_values, c):
    # Calculate scale and shape parameters for the entire mu tensor
    c0, c1, c2 = c
    #c0, c1 = c
    scale = c0 + c1 * mu + c2 * mu**2  # shape: (100,)
    shape = mu + 1  # Gamma shape parameter (alpha), shape: (100,)
    
    # Ensure scale is valid
    scale = torch.clamp(scale, min=1e-6)

    # Expand bin centers and widths for batch processing
    bin_centers = bin_centers[None, :]  # shape: (1, num_bins)
    bin_widths = bin_widths[None, :]    # shape: (1, num_bins)
    
    # Create batched Gamma distributions
    #print(f"{bin_centers.shape=}")
    #print(f"{shape=}")
    #print(f"{scale=}")
    gamma_dists = Gamma(shape[:, None], scale[:, None])  # Broadcasting shape: (100, num_bins)
    pdf_values = gamma_dists.log_prob(bin_centers).exp() * bin_widths  # shape: (100, num_bins)
    #print(pdf_values)
    #print(f"{pdf_values.shape=}")
    #print(f"{hist_values.shape=}")
    
    # Convert hist_values to tensor and reshape for batch operations
    hist_values = hist_values.to(torch.float64)  # shape: (100, num_bins)
    
    # Compute negative log-likelihood for all batches
    nll = -torch.sum(hist_values * torch.log(pdf_values) - pdf_values, dim=1)
    #print(nll)
    
    return torch.sum(nll)

# Constants and inputs
c = torch.tensor([1.9935, 0.735, 0.0365])
mu_all = torch.ones(50, dtype=torch.float64)*5  # Batch of 100 mu values
optimizer = torch.optim.Adam([mu_all, c], lr=1e-2)

# Prepare bin centers, widths, and histograms
bin_centers = torch.tensor(nc["diameter_bin_center"][:][:20], dtype=torch.float64)  # shape: (num_bins,)
bin_widths = torch.tensor(nc["diameter_bin_width"][:][:20], dtype=torch.float64)   # shape: (num_bins,)
hist_values = torch.tensor(np.sum(nc["raw_drop_number"][:], axis=2).astype(np.float64)[:50, :20], dtype=torch.float64)  # shape: (100, num_bins)

# Compute loss vectorized
loss = compute_negative_log_likelihood_vectorized(mu_all, bin_centers, bin_widths, hist_values, c)
print(loss)

In [ ]:
# case 1

In [ ]:
c = torch.nn.Parameter(torch.tensor([1.9935, 0.735, 0.0365], dtype=torch.float32, requires_grad=True))
mu_all = torch.nn.Parameter(5*torch.ones(80, dtype=torch.float32, requires_grad=True))

bin_centers = torch.tensor(nc["diameter_bin_center"][:][:20], dtype=torch.float32) 
bin_widths = torch.tensor(nc["diameter_bin_width"][:][:20], dtype=torch.float32) 
hist_values = torch.tensor(np.sum(nc["raw_drop_number"][:], axis=2).astype(np.float32)[:80, :20], dtype=torch.float32)  # shape: (100, num_bins)

optimizer = torch.optim.Adam([mu_all, c], lr=1e-2)

for epoch in range(50000):
    optimizer.zero_grad()
    loss = compute_negative_log_likelihood_vectorized(mu_all, bin_centers, bin_widths, hist_values, c)
    loss.backward()
    optimizer.step()

    if epoch % 5000 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}, c0: {c[0].item()}, c1: {c[1].item()}, c2: {c[2].item()}")

    #print(c.grad)
    #print(mu_all.grad)
    #print(loss)
    
print(c)

In [ ]:
# case 2

In [ ]:
c = torch.nn.Parameter(torch.tensor([1.9935, 0.735, 0.0365], dtype=torch.float32, requires_grad=True))
mu_all = torch.nn.Parameter(5*torch.ones(100, dtype=torch.float32, requires_grad=True))

bin_centers = torch.tensor(nc["diameter_bin_center"][:][4:20], dtype=torch.float32) 
bin_widths = torch.tensor(nc["diameter_bin_width"][:][4:20], dtype=torch.float32)
hist_values = torch.tensor(np.sum(nc["raw_drop_number"][:], axis=2).astype(np.float32)[:100, 4:20], dtype=torch.float32)  # shape: (100, num_bins)

optimizer = torch.optim.Adam([mu_all, c], lr=1e-2)

for epoch in range(50000):
    optimizer.zero_grad()
    loss = compute_negative_log_likelihood_vectorized(mu_all, bin_centers, bin_widths, hist_values, c)
    loss.backward()
    optimizer.step()

    if epoch % 5000 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}, c0: {c[0].item()}, c1: {c[1].item()}, c2: {c[2].item()}")

    #print(c.grad)
    #print(mu_all.grad)
    #print(loss)
    
print(c)

In [ ]:
hist_values

In [ ]:
bin_centers

In [ ]:
hist_values_test.shape

In [ ]:
pdf_values

In [ ]:
hist_values_test = torch.tensor(np.sum(nc["raw_drop_number"][:], axis=2).astype(np.float32)[80:, 4:20], dtype=torch.float32)  # shape: (100, num_bins)

def negative_log_likelihood_constrained_scalar2(shape):
    #if shape <= 0:
    #    return np.inf
    mu = shape - 1
    scale = 1/(c0 + c1 * mu + c2 * mu**2)
    if scale <= 1e-6:  # Avoid scale being too small
        return np.inf
    pdf_values = gamma.pdf(bin_centers, a=shape, scale=scale) * bin_widths
    print(pdf_values)
    #pdf_values = np.maximum(pdf_values, 1e-100)
    return -np.sum(hist_values_test * np.log(pdf_values) - pdf_values)

negative_log_likelihood_constrained_scalar2(2)

#compute_negative_log_likelihood_vectorized(mu_all, bin_centers, bin_widths, hist_values_test, c)

In [ ]:
Lambda_all = c[0]+c[1]*mu_all#+c[2]*mu_all*mu_all

for i in range(100):
    print(f"{i=}")
    hist_values = np.sum(nc["raw_drop_number"][:], axis=2)[i][4:20].astype(np.float32)
    hist_values_normalized = hist_values / (hist_values.sum() * bin_widths)
    #hist_values = np.random.poisson(10, size=len(bin_edges) - 1)

    bin_centers = torch.tensor(np.array(nc["diameter_bin_center"][:][4:20]))
    bin_widths = torch.tensor(np.array(nc["diameter_bin_width"][:][4:20]))
    
    x_values = np.linspace(0, 5, 501)
    #shape = 8
    mu = mu_all[i].detach()
    Lambda = Lambda_all[i].detach()
    shape = mu + 1
    scale = Lambda
    fitted_pdf = Gamma(shape, scale).log_prob(x_values).exp()
    #fitted_pdf2 = gamma.pdf(x_values, a=fitted_shape2, scale=fitted_scale2)

    plt.figure(figsize=(8, 5))
    plt.bar(bin_centers, hist_values_normalized, width=bin_widths, alpha=0.5, label="Histogram")
    plt.plot(x_values, fitted_pdf, label=f"Fitted Gamma PDF\n(shape={shape:.2f}, scale={scale:.2f})", color="red")
    #plt.plot(x_values, fitted_pdf2, label=f"Fitted Gamma PDF\n(shape={fitted_shape2:.2f}, scale={fitted_scale2:.2f})", color="blue")
    #plt.annotate(f"{D1=:.2f} mm", [0.7, 0.65], xycoords='axes fraction', color="red")
    #plt.annotate(f"{D2=:.2f} mm", [0.7, 0.6], xycoords='axes fraction', color="blue")
    plt.xlim(0, 5)
    plt.xlabel("Value")
    plt.ylabel("Density")
    plt.legend()
    plt.title("EPFL, COMMON 2011")
    #plt.show()
    plt.savefig(f"step{i}_newfit")

In [ ]:
D0 = (mu_all+3.67)/Lambda_all
mappable = plt.scatter(mu_all.detach(), Lambda_all.detach(), c=D0.detach())
plt.colorbar(mappable)

In [ ]:
from torch.distributions import Gamma
import torch
import numpy as np
import netCDF4
import glob
import pandas as pd
campaign = "COMMON_2011"
fileall = glob.glob(f"../disdrodb-data/DISDRODB/Processed/EPFL/{campaign}/L0B/??/L0B.{campaign}.??.*.V0.nc")
infile = fileall[2]
nc = netCDF4.Dataset(infile,"r")

def compute_negative_log_likelihood(mu, bin_centers, bin_widths, hist_values, c0, c1, c2):
    scale = c0+c1*mu+c2*mu*mu
    shape = mu + 1  # shape = alpha
    if scale <= 1e-6:
        return torch.tensor(float('inf'), dtype=torch.float64)  # Avoid invalid scale
    
    # Use PyTorch Gamma distribution
    gamma_dist = Gamma(shape, scale)  # PyTorch uses rate = 1/scale
    #print(gamma_dist.log_prob(torch.tensor(bin_centers, dtype=torch.float64)).exp())
    #print(torch.from_numpy(np.array(bin_widths)))
    pdf_values = gamma_dist.log_prob(torch.tensor(bin_centers, dtype=torch.float64)).exp() * torch.from_numpy(np.array(bin_widths))
    hist_values = torch.tensor(np.array(hist_values).astype(np.float64), dtype=torch.float64)
    #print(pdf_values)

    # Negative log-likelihood
    return -torch.sum(hist_values * torch.log(pdf_values) - pdf_values)

campaign = "COMMON_2011"
fileall = glob.glob(f"../disdrodb-data/DISDRODB/Processed/EPFL/{campaign}/L0B/??/L0B.{campaign}.??.*.V0.nc")
infile = fileall[2]
nc = netCDF4.Dataset(infile,"r")

c0, c1, c2 = [1.9935, 0.735, 0.0365]
mu_all = torch.ones(100)
loss = 0
for i in range(100):
    lossnow = compute_negative_log_likelihood(
        mu=mu_all[i], 
        bin_centers=np.array(nc["diameter_bin_center"][:][4:20]), 
        bin_widths=np.array(nc["diameter_bin_width"][:][4:20]), 
        hist_values=np.array(np.sum(nc["raw_drop_number"][:], axis=2)[i][4:20]), 
        c0=c0, 
        c1=c1, 
        c2=c2
    )
    loss = loss + lossnow
    print(lossnow)
    
print(loss)

In [ ]:
campaign = "COMMON_2011"
fileall = glob.glob(f"../disdrodb-data/DISDRODB/Processed/EPFL/{campaign}/L0B/??/L0B.{campaign}.??.*.V0.nc")
infile = fileall[2]
nc = netCDF4.Dataset(infile,"r")

def compute_negative_log_likelihood(mu, bin_centers, bin_widths, hist_values, c0, c1, c2):
    scale = c0+c1*mu+c2*mu*mu
    shape = mu + 1  # shape = alpha
    if scale <= 1e-6:
        return torch.tensor(float('inf'), dtype=torch.float64)  # Avoid invalid scale
    
    # Use PyTorch Gamma distribution
    gamma_dist = Gamma(shape, scale)  # PyTorch uses rate = 1/scale
    #print(gamma_dist.log_prob(torch.tensor(bin_centers, dtype=torch.float64)).exp())
    #print(torch.from_numpy(np.array(bin_widths)))
    pdf_values = gamma_dist.log_prob(torch.tensor(bin_centers, dtype=torch.float64)).exp() * torch.from_numpy(np.array(bin_widths))
    hist_values = torch.tensor(np.array(hist_values).astype(np.float64), dtype=torch.float64)
    #print(pdf_values)

    # Negative log-likelihood
    return -torch.sum(hist_values * torch.log(pdf_values) - pdf_values)

campaign = "COMMON_2011"
fileall = glob.glob(f"../disdrodb-data/DISDRODB/Processed/EPFL/{campaign}/L0B/??/L0B.{campaign}.??.*.V0.nc")
infile = fileall[2]
nc = netCDF4.Dataset(infile,"r")

c0, c1, c2 = [1.9935, 0.735, 0.0365]
mu_all = torch.ones(100)
loss = 0
for i in range(100):
    lossnow = compute_negative_log_likelihood(
        mu=mu_all[i], 
        bin_centers=np.array(nc["diameter_bin_center"][:][4:20]), 
        bin_widths=np.array(nc["diameter_bin_width"][:][4:20]), 
        hist_values=np.array(np.sum(nc["raw_drop_number"][:], axis=2)[i][4:20]), 
        c0=c0, 
        c1=c1, 
        c2=c2
    )
    loss = loss + lossnow
    print(lossnow)
    
print(loss)

In [ ]:
from torch.distributions import Gamma
import torch
import numpy as np
import netCDF4
import glob

campaign = "COMMON_2011"
fileall = glob.glob(f"../disdrodb-data/DISDRODB/Processed/EPFL/{campaign}/L0B/??/L0B.{campaign}.??.*.V0.nc")
infile = fileall[2]
nc = netCDF4.Dataset(infile, "r")

def compute_negative_log_likelihood_vectorized(mu, bin_centers, bin_widths, hist_values, c0, c1, c2):
    # Calculate scale and shape parameters for the entire mu tensor
    scale = c0 + c1 * mu + c2 * mu**2  # shape: (100,)
    shape = mu + 1  # Gamma shape parameter (alpha), shape: (100,)
    
    # Ensure scale is valid
    scale = torch.clamp(scale, min=1e-6)

    # Expand bin centers and widths for batch processing
    bin_centers = bin_centers[None, :]  # shape: (1, num_bins)
    bin_widths = bin_widths[None, :]    # shape: (1, num_bins)
    
    # Create batched Gamma distributions
    #print(f"{bin_centers.shape=}")
    #print(f"{shape=}")
    #print(f"{scale=}")
    gamma_dists = Gamma(shape[:, None], scale[:, None])  # Broadcasting shape: (100, num_bins)
    pdf_values = gamma_dists.log_prob(bin_centers).exp() * bin_widths  # shape: (100, num_bins)
    #print(pdf_values)
    #print(f"{pdf_values.shape=}")
    #print(f"{hist_values.shape=}")
    
    # Convert hist_values to tensor and reshape for batch operations
    hist_values = hist_values.to(torch.float64)  # shape: (100, num_bins)
    
    # Compute negative log-likelihood for all batches
    nll = -torch.sum(hist_values * torch.log(pdf_values) - pdf_values, dim=1)
    
    return torch.sum(nll)

# Constants and inputs
c0, c1, c2 = [1.9935, 0.735, 0.0365]
mu_all = torch.ones(100, dtype=torch.float64)  # Batch of 100 mu values

# Prepare bin centers, widths, and histograms
bin_centers = torch.tensor(nc["diameter_bin_center"][:][4:20], dtype=torch.float64)  # shape: (num_bins,)
bin_widths = torch.tensor(nc["diameter_bin_width"][:][4:20], dtype=torch.float64)   # shape: (num_bins,)
hist_values = torch.tensor(np.sum(nc["raw_drop_number"][:], axis=2).astype(np.float64)[:100, 4:20], dtype=torch.float64)  # shape: (100, num_bins)

# Compute loss vectorized
loss = compute_negative_log_likelihood_vectorized(mu_all, bin_centers, bin_widths, hist_values, c0, c1, c2)
print(loss)


In [ ]:
from torch.distributions import Gamma
import torch
import netCDF4
import glob
import numpy as np

# Function to compute the negative log-likelihood
def compute_negative_log_likelihood(mu, bin_centers, bin_widths, hist_values, c0, c1, c2):
    scale = c0 + c1 * mu + c2 * mu * mu
    shape = mu + 1  # shape = alpha
    # Ensure scale > 0
    scale = torch.clamp(scale, min=1e-6)  

    # PyTorch Gamma distribution
    gamma_dist = Gamma(shape, 1 / scale)  # PyTorch uses rate = 1/scale
    pdf_values = gamma_dist.log_prob(bin_centers).exp() * bin_widths

    # Negative log-likelihood
    return -torch.sum(hist_values * torch.log(pdf_values) - pdf_values)

# Load data
campaign = "COMMON_2011"
fileall = glob.glob(f"../disdrodb-data/DISDRODB/Processed/EPFL/{campaign}/L0B/??/L0B.{campaign}.??.*.V0.nc")
infile = fileall[2]
nc = netCDF4.Dataset(infile, "r")

# Convert data to torch tensors
bin_centers = torch.tensor(nc["diameter_bin_center"][:], dtype=torch.float64)
bin_widths = torch.tensor(nc["diameter_bin_width"][:], dtype=torch.float64)
hist_values_all = torch.tensor(np.sum(nc["raw_drop_number"][:], axis=2).astype(np.float64), dtype=torch.float64)

# Initialize parameters to optimize
mu_all = 5*torch.ones(100, dtype=torch.float64, requires_grad=True)
c0 = torch.tensor(1.9935, dtype=torch.float64, requires_grad=True)
c1 = torch.tensor(0.735, dtype=torch.float64, requires_grad=True)
c2 = torch.tensor(0.0365, dtype=torch.float64, requires_grad=True)

# Optimizer
optimizer = torch.optim.Adam([mu_all, c0, c1, c2], lr=1e-2)

# Optimization loop
num_epochs = 5000
for epoch in range(num_epochs):
    optimizer.zero_grad()
    loss = 0
    for i in range(100):
        loss = loss + compute_negative_log_likelihood(
            mu=mu_all[i],
            bin_centers=bin_centers,
            bin_widths=bin_widths,
            hist_values=hist_values_all[i],
            c0=c0,
            c1=c1,
            c2=c2
        )
    
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}, c0: {c0.item()}, c1: {c1.item()}, c2: {c2.item()}")

print("Optimization finished.")
print(f"Optimized c0: {c0.item()}, c1: {c1.item()}, c2: {c2.item()}")
print(f"First 10 optimized mu values: {mu_all[:10].detach().numpy()}")


In [ ]:
c0